# ZINC Streaming Demo

This notebook showcases the schema-frozen streaming training path for `ConditionalNodeFieldGraphGenerator`.

- source: raw ZINC CSV
- warmup: first 1000 accepted graphs
- stream limit: `0.1`
- targets: none
- final cells: generate 7 graphs without feasibility filtering, then 7 with filtering


In [1]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

from pathlib import Path
import random

import numpy as np
from IPython.core.display import HTML

HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell; text-align: center; vertical-align: middle;}</style>')

from conditional_node_field_graph_generator.notebooks import configure_notebook
globals().update(configure_notebook(require_nsppk=True, print_torch=True))

from abstractgraph_graphicalizer.chem import download_zinc_dataset, draw_molecules
from conditional_node_field_graph_generator.extensions.demo import show_molecules
from conditional_node_field_graph_generator.extensions.demo.pipeline import build_graph_generator


PyTorch version: 2.2.2
CUDA available: False
Enabling RDKit 2025.09.3 jupyter extensions


/Users/fabriziocosta/miniconda3/envs/py311/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.2.0)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [2]:
RANDOM_SEED = 7
STREAM_LIMIT = 0.01
WARMUP_SIZE = 2048
STREAM_BATCH_SIZE = 32
MAXIMUM_EPOCHS = 100
EMBEDDING_DIM = 64
MODEL_NAME = f'zinc-streaming-n{EMBEDDING_DIM}-s{STREAM_LIMIT}-w{WARMUP_SIZE}-b{STREAM_BATCH_SIZE}-e{MAXIMUM_EPOCHS}'
ZINC_DATA_ROOT = NOTEBOOK_DATA_ROOT / 'zinc'
DECODER_N_JOBS = -1

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [3]:
csv_path = download_zinc_dataset(ZINC_DATA_ROOT)
print(f'ZINC CSV: {csv_path}')

graph_generator = build_graph_generator(
    latent_embedding_dimension=EMBEDDING_DIM,
    number_of_transformer_layers=2,
    transformer_attention_head_count=4,
    maximum_epochs=MAXIMUM_EPOCHS,
    batch_size=STREAM_BATCH_SIZE,
    verbose=1,
    decoder_n_jobs=DECODER_N_JOBS,
    artifact_root=ARTIFACT_ROOT,
    checkpoint_root=CHECKPOINT_ROOT,
    model_name=MODEL_NAME,
    model_dir=SAVED_GENERATOR_ROOT,
)
graph_generator.graph_decoder.diagnostic_graph_renderer = draw_molecules


ZINC CSV: /Users/fabriziocosta/Resilio Sync/Sync/Projects/NodeField/notebooks/datasets/zinc/zinc_250k.csv
Configured graph generator model_name=zinc-streaming-n64-s0-01-w2048-b32-e100 model_dir=/Users/fabriziocosta/Resilio Sync/Sync/Projects/NodeField/.artifacts/saved_generators


In [ ]:
graph_generator.fit_from_stream(
    csv_path,
    'zinc_csv',
    warmup_size=WARMUP_SIZE,
    batch_size=STREAM_BATCH_SIZE,
    limit=STREAM_LIMIT,
    random_state=RANDOM_SEED,
    verbose=True,
)

print('stream_seen_ =', graph_generator.stream_seen_)
print('stream_warmup_count_ =', graph_generator.stream_warmup_count_)
print('stream_training_seen_ =', graph_generator.stream_training_seen_)
print('stream_training_accepted_ =', graph_generator.stream_training_accepted_)
print('stream_training_skipped_ =', graph_generator.stream_training_skipped_)
print('stream_acceptance_rate_ =', graph_generator.stream_acceptance_rate_)


In [ ]:
raw_samples = graph_generator.sample(
    n_samples=7,
    apply_feasibility_filtering=False,
)
show_molecules(raw_samples, n=7, title='Streaming ZINC samples without feasibility filtering')


In [ ]:
if graph_generator.feasibility_estimator is None:
    raise RuntimeError('Feasibility estimator is unavailable in this environment.')

filtered_samples = graph_generator.sample(
    n_samples=7,
    apply_feasibility_filtering=True,
)
show_molecules(filtered_samples, n=7, title='Streaming ZINC samples with feasibility filtering')
